In [17]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import operator

load_dotenv()


True

In [18]:
model = ChatGroq(
    model="openai/gpt-oss-120b"
)


In [19]:
class Formant(BaseModel):
    feedback : str = Field (description="detailed feeback on essay")
    rating : int = Field(description="give a rating out of 10",le=10,ge=1)
    

In [20]:
structedmodel = model.with_structured_output(Formant)

In [21]:
essay = """ Artificial Intelligence (AI) is one of the most important technological innovations of the modern world. It refers to the ability of machines and computer systems to perform tasks that normally require human intelligence, such as learning, reasoning, problem-solving, understanding language, and making decisions. AI has become an essential part of our daily lives, transforming industries such as healthcare, education, banking, transportation, and entertainment.
What is Artificial Intelligence?
Artificial Intelligence is a branch of computer science that focuses on developing intelligent systems capable of performing tasks with minimal human intervention. AI systems use data, algorithms, and computational techniques to identify patterns, make predictions, and generate useful outputs.
For example, virtual assistants like Siri and Alexa use AI to understand voice commands, while applications like ChatGPT use advanced language models to generate human-like responses. Recommendation systems used by YouTube, Netflix, and Amazon also rely on AI to suggest content and products based on user preferences. """
prompt = f"analyse the eassy and give a feedback and rating on eassy {essay}"

In [22]:
structedmodel.invoke(prompt)

Formant(feedback='The essay provides a concise introduction to artificial intelligence, covering its definition, core capabilities, and everyday applications. It is well‑structured, beginning with a broad statement about AI’s importance, followed by a clear definition, and then concrete examples that illustrate how AI is embedded in modern life. The language is generally clear and accessible, making the piece suitable for a general audience.\n\n**Strengths**\n- **Clear organization**: The essay follows a logical flow (importance → definition → examples).\n- **Relevant examples**: Mentioning Siri, Alexa, ChatGPT, and recommendation systems grounds the abstract concept in familiar technology.\n- **Broad coverage**: It touches on multiple domains (healthcare, education, banking, transportation, entertainment), highlighting AI’s pervasive impact.\n\n**Areas for improvement**\n1. **Depth and specificity**: The essay stays at a high level. Adding a brief explanation of how AI techniques (e.g

In [23]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [ ]:
def evaluate_language(state: UPSCState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structedmodel.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [25]:
def evaluate_analysis(state:UPSCState):

    prompt=f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structedmodel.invoke(prompt)

    return {'analysis_feedback':output.feedback,'individual_scores': [output.score]}

In [26]:
def evaluate_thought(state: UPSCState):

    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structedmodel.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}